In [155]:
import os
import re
import sys
from pathlib import Path
from typing import List, Dict
from langchain_core.prompts import ChatPromptTemplate
import re
import numpy as np
import pandas as pd
import requests
import rapidfuzz
import json
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import SentenceTransformer
sys.path.append(str(Path.cwd().parent))  # Add parent directory to sys.path for local imports

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from agent.agents.explainer import explain_rule
from agent.tools.mvel_parser_tool import parse_mvel_branches
from agent.llm import get_llm

In [156]:
EXPLAIN_PROMPT = ChatPromptTemplate.from_messages([
        (
            "system",
            "You write for non-technical stakeholders. "
            "Do not mention code, syntax, or programming terms."
        ),
        (
            "user",
            "Convert the following extracted rule structure into clear English.\n\n"
            "Requirements:\n"
            "- Start with a 1–2 sentence Summary\n"
            "- Then list Decision logic as bullet points\n"
            "- One bullet per branch, in order\n"
            "- Use 'Otherwise,' for the DEFAULT branch\n"
            "- Use provided context to define business terms if relevant\n\n"
            "Context:\n"
            "{context}\n\n"     
            "Rule extraction (JSON):\n"
            "{extraction_json}"
        )
    ])

# NO PARSING
EXPLAIN_V2 = ChatPromptTemplate.from_messages([
        (
            "system",
            "You write for non-technical stakeholders. "
            "Do not mention code, syntax, or programming terms."
        ),
        (
            "user",
            "Convert the following MVEL text into clear, natural English.\n"
            "Explain what the expression does in plain language.\n"
            "Break down each part of the expression step by step.\n"
            "If there are conditions, operators, or functions, describe what they mean.\n"
            "Write the description in sentences. \n"
            "Provide the final meaning as if explaining to someone with no programming background.\n"
            "MVEL TEXT:\n"
            "{mvel_text}"
        )
    ])



ENGLISH_TO_MVEL = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in writing MVEL rules for business systems. "
        "Generate accurate, production-ready MVEL expressions."
    ),
    (
        "user",
        "Convert the following English requirement into a valid MVEL expression.\n"
        "Translate the business logic precisely.\n"
        "Use proper MVEL syntax.\n"
        "Assume referenced variables already exist unless stated otherwise.\n"
        "Add null checks where appropriate.\n"
        "Return ONLY the MVEL expression.\n"
        "Do not include explanations.\n"
        "ENGLISH REQUIREMENT:\n"
        "Format the output as a single boolean MVEL expression similar to this pattern:\n(input.message != null && input.message.equalsIgnoreCase('MT')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.clientId == null || input.clientId == 'N')\nUse grouped conditions with parentheses and logical operators (&&, ||) exactly in this style."
        "{english_text}"
    )
])



In [157]:
# Initialize an LLM and provide well-named helper functions for the notebook
# Keeps functions explicit and easy for users to call.
llm = get_llm(model="llama3.1", temperature=0.0)

def explain_parsed_rule(rule_text: str, context: str = "Business context or glossary here") -> str:
    """Parse MVEL text, run the explain prompt pipeline, and return English.
    Uses the parsed extraction JSON as the prompt input.
    """
    parsed = parse_mvel_branches(rule_text)
    extraction_json = json.dumps(parsed, ensure_ascii=False, indent=2)
    chain = EXPLAIN_PROMPT | llm | StrOutputParser()
    print("[explain_parsed_rule] Running explanation chain on parsed extraction...")
    explanation = chain.invoke({"extraction_json": extraction_json, "context": context}).strip()
    print("[explain_parsed_rule] Completed.")
    return explanation

def explain_raw_mvel(rule_text: str) -> str:
    """Explain raw MVEL text without parsing. Useful when parser fails.
    """
    chain = EXPLAIN_V2 | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    print("[explain_raw_mvel] Completed.")
    return response.content

def convert_english_to_mvel(english_text: str) -> str:
    """Convert a plain-English requirement into an MVEL expression via the LLM.
    Returns the raw string returned by the model (intended to be a single MVEL expression).
    """
    chain = ENGLISH_TO_MVEL | llm
    print("[convert_english_to_mvel] Converting English to MVEL...")
    response = chain.invoke({"english_text": english_text})
    print("[convert_english_to_mvel] Completed.")
    return response.content

In [158]:
# Utilities for fuzzy matching and evaluation against a CSV library of rules
import re
import pandas as pd
from rapidfuzz import fuzz

CSV_PATH = "pre_match.csv"

# Example test MVELs (replace or extend as needed)
tests = [
    "(input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')",
    "(input.message != null && input.message.equalsIgnoreCase('ABC')) && (input.estimate != null && input.estimate.equalsIgnoreCase('money')) && (input.srcworkType != null && input.srcworkType.equalsIgnoreCase('normal')) && (input.clientId != null && input.clientId != 'Y') && (input.transactionIndex == 'N') && (input.Ref.contains('CASH MONEY HEROES'))",
    "(input.message != null && (input.message.equalsIgnoreCase('FT900') || input.messageTypeCode.equalsIgnoreCase('FT500') || input.message.equalsIgnoreCase('SSN') || input.message.equalsIgnoreCase('MT'))) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transactionCode.equalsIgnoreCase('CAPITAL HILL')) && (input.entityFlag != null && input.entityFlag == 'Y')",
]

english_texts = []  # populated by evaluate

def normalize(text: str) -> str:
    t = str(text).strip()
    t = re.sub(r"\s+", " ", t)
    t = t.replace(" | | ", " || ")
    return t

def similarity(a: str, b: str) -> float:
    # Returns 0..1 similarity score using rapidfuzz token_set_ratio
    return fuzz.token_set_ratio(normalize(a), normalize(b)) / 100.0

def best_target_match(model_output: str, library_targets: list[str]) -> tuple[int, float]:
    scores = []
    for t in library_targets:
        sim = similarity(model_output, t)
        scores.append(sim)
    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    print("generated output:", model_output)
    print("matched label:", library_targets[best_idx])
    print(float(scores[best_idx]))
    return best_idx, float(scores[best_idx])

def evaluate(explain_fn):
    """Evaluate a given explanation function against the CSV library.
    explain_fn: callable that accepts an MVEL string and returns English text.
    """
    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=["RULE", "TARGET"]).reset_index(drop=True)

    library_rules = df["RULE"].astype(str).tolist()
    library_targets = df["TARGET"].astype(str).tolist()

    print(f"Loaded library rows: {len(library_rules)}")

    for i, test_rule in enumerate(tests, start=1):
        print("\n" + "=" * 80)
        print(f"TEST {i}")
        print("TEST RULE:\n", test_rule)

        # Call the provided explain function
        model_output = explain_fn(test_rule)
        english_texts.append(model_output)

        best_idx, best_score = best_target_match(model_output, library_targets)


def eval_convert(conversion):
    df = pd.read_csv(CSV_PATH)
    library_rules = df["RULE"].astype(str).tolist()
    explanation = df["TARGET"].astype(str).tolist()

    for idx, rule in enumerate(english_texts):
        # Call conversion and normalize to a single-line MVEL string
        output = conversion(rule)
        output_str = str(output)
        clean_rule = output_str.replace('`', '').replace('\n', ' ').strip()

        # Find best match in the dataset (compare converted MVEL to library rules)
        best_idx, best_score = best_target_match(clean_rule, library_targets=library_rules)



In [159]:
# Run evaluation using the parsed-rule explainer (preferred when parser works)
evaluate(explain_fn=explain_parsed_rule)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_parsed_rule] Running explanation chain on parsed extraction...
[explain_parsed_rule] Completed.
generated output: Here is the extracted rule structure rewritten in clear English:

**Summary**
This rule checks if a message meets certain criteria and advises on whether to proceed with a transaction.

**Decision Logic**

* If the input message contains 'MT', 'saipan', or 'CA' (case-insensitive) and the money code is 'Actual', then check the next condition.
* If the transaction type is 'CAPITAL HILL', then check the final condition.
* Otherwise, d

In [160]:
# After you have run `evaluate` to populate `english_texts`,
# you can attempt to convert the generated English back to MVEL and compare.
eval_convert(conversion=convert_english_to_mvel)

[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && ((input.transactionType != null && input.transactionType.equalsIgnoreCase('CAPITAL HILL')) || input.clientId == null || input.clientId == 'N')
matched label: (input.message != null && input.message.equalsIgnoreCase('MT')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.clientId == null || input.clientId == 'N')
0.8029197080291971
[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('ABC')) && (input.estimate != null && input.estimate.equalsIgnoreCase('money')) && (input.sourceWorkType != null && input.sourceWo